# Data Cleaning Review: Gaps in Notebook 02 and What Can Be Cleaned

This notebook evaluates `02_natural_language_data_prep.ipynb`, runs the data-quality checks it did not cover, and turns each finding into a cleaning proposal with evidence and row counts. It then builds a cleaned working table in which only approved proposals take effect.

## Natural-language request

> Evaluate notebook 02 and find out what can be done to clean the data. Do it in a separate notebook.

## Ground rules

- The raw file stays unchanged. All work happens on a copy.
- Any check that reads `is_canceled`, `reservation_status`, or `reservation_status_date` uses **train only**. Choose and proof are checked only for structural problems (for example zero nights or a negative ADR), never for outcomes.
- Adding the contract's `nights` column and the `flag_...` columns is not cleaning: no value changes and no row is removed. Every recode or row decision waits for your approval in section 10, as in notebook 02.
- `src/contract.py` is used unchanged.

## 1. Evaluation of notebook 02

Notebook 02 kept the raw file unchanged, used the contract split, kept proof outcomes sealed, and excluded outcome fields from predictors. Those parts are sound. Its checks have these gaps:

| Area | What notebook 02 did | Gap | Covered here |
|---|---|---|---|
| Missing values | Counted `NaN` | Did not look for codes that mean "missing" or "none": `meal = Undefined`, `market_segment` / `distribution_channel = Undefined`, and the 2-letter country code `CN`. Gate D2 calls NULL `agent` / `company` "missing", but the dataset article defines NULL there as "not applicable". | §3 |
| Types | Listed dtypes | `agent` and `company` IDs and the `children` count are stored as decimals because of NULL / NaN. | §2 |
| Ranges | Checked negatives in train only | The only negative ADR is in proof. Structural checks on choose and proof are allowed as long as no outcome is read. | §4 |
| Cross-field logic | None | Zero-guest and zero-night bookings, children without adults, placeholder room type `P`, target vs reservation status, status date vs stay, week number vs date. | §4, §5 |
| Duplicates | Counted 16,291 | Not characterised: which segments they come from, how they relate to cancellation, and what removing them would do to the training rate and to the scored book. | §6 |
| Candidate fields | Missingness only | No check of `is_repeated_guest` against the prior-booking counts, and no check that the counts look alike in train, choose, and proof. | §7 |
| Scoring impact | Not considered | `contract.nightly_table` needs a `nights` column, and any row removal or ADR change alters a hotel-night's book or stay value. | §8 |

## Setup: load, derive the contract columns, and split

`nights` is the contract's definition (`stays_in_weekend_nights + stays_in_week_nights`). `guests` is used only for validity checks. Duplicate checks on choose and proof compare the non-outcome columns, so no proof outcome is read.

In [1]:
from pathlib import Path
import sys

# Find the repository so this works from the repo folder or notebooks/.
repo_root = None
for folder in (Path.cwd(), *Path.cwd().parents):
    if (folder / "src" / "data.py").exists():
        repo_root = folder
        break

if repo_root is None:
    raise FileNotFoundError("Could not find the project src/ folder.")

source_folder = str(repo_root / "src")
if source_folder not in sys.path:
    sys.path.insert(0, source_folder)

import contract
import data
import pandas as pd

raw_bookings = data.read_raw()
source_columns = list(raw_bookings.columns)
outcome_columns = ["is_canceled", "reservation_status", "reservation_status_date"]
structural_columns = [column for column in source_columns if column not in outcome_columns]

bookings = raw_bookings.copy()
bookings["arrival_date"] = data.arrival_date(bookings)
bookings["nights"] = bookings["stays_in_weekend_nights"] + bookings["stays_in_week_nights"]
bookings["guests"] = bookings["adults"] + bookings["children"].fillna(0) + bookings["babies"]

train, choose, proof = contract.split_periods(bookings)
splits = {"train": train, "choose": choose, "proof": proof}


def count_by_split(checks):
    """Count structural checks in each period without reading outcome columns."""
    rows = []

    for check_name, check in checks.items():
        row = {}
        for period_name, period in splits.items():
            row[period_name] = int(check(period).sum())
        rows.append(pd.Series(row, name=check_name))

    return pd.DataFrame(rows)


print(f"Rows: {len(raw_bookings):,}; source columns: {len(source_columns)}")
print("Rows per period:", {name: len(period) for name, period in splits.items()})

Rows: 119,390; source columns: 32
Rows per period: {'train': 49157, 'choose': 29546, 'proof': 40687}


## 2. Column types

NULL `agent` / `company` values and 4 missing `children` values were read as `NaN`, which forces whole-number columns to decimals. Proposal C2 stores the IDs as text; proposal C4 fills the 4 missing counts.

In [2]:
type_columns = ["agent", "company", "children", "reservation_status_date"]
type_reasons = [
    "Agency ID; NULL read as NaN forces decimals",
    "Company ID; NULL read as NaN forces decimals",
    "Count; 4 NaN rows force decimals",
    "Date stored as text (outcome field, never a predictor)",
]
type_issues = pd.DataFrame({
    "column": type_columns,
    "current_type": [str(raw_bookings[column].dtype) for column in type_columns],
    "reason": type_reasons,
})
print(type_issues.to_string(index=False))

for column in ["agent", "company"]:
    whole_numbers = bool(raw_bookings[column].dropna().mod(1).eq(0).all())
    print(f"{column}: every non-missing ID is a whole number: {whole_numbers}")

                 column current_type                                                 reason
                  agent      float64            Agency ID; NULL read as NaN forces decimals
                company      float64           Company ID; NULL read as NaN forces decimals
               children      float64                       Count; 4 NaN rows force decimals
reservation_status_date          str Date stored as text (outcome field, never a predictor)
agent: every non-missing ID is a whole number: True
company: every non-missing ID is a whole number: True


## 3. Hidden missing values and duplicate codes

The dataset article (Antonio et al., 2019) defines `meal` as "Undefined/SC - no meal package" and says NULL in `agent` / `company` means "not applicable". These counts use all three periods but read no outcome column.

In [3]:
code_checks = {
    "meal = Undefined": lambda d: d["meal"].eq("Undefined"),
    "meal = SC": lambda d: d["meal"].eq("SC"),
    "market_segment = Undefined": lambda d: d["market_segment"].eq("Undefined"),
    "distribution_channel = Undefined": lambda d: d["distribution_channel"].eq("Undefined"),
    "agent NULL (no agent)": lambda d: d["agent"].isna(),
    "company NULL (no company)": lambda d: d["company"].isna(),
    "country missing": lambda d: d["country"].isna(),
    "country = CN (2-letter code)": lambda d: d["country"].eq("CN"),
    "country = CHN": lambda d: d["country"].eq("CHN"),
    "children missing": lambda d: d["children"].isna(),
}
print(count_by_split(code_checks).to_string())

print("\nMeal codes by hotel (all periods):")
print(pd.crosstab(bookings["meal"], bookings["hotel"]).to_string())

code_lengths = bookings["country"].dropna().str.len().value_counts().sort_index()
print("\nCountry code lengths:", code_lengths.to_dict())

                                  train  choose  proof
meal = Undefined                    548     208    413
meal = SC                          2471    3001   5178
market_segment = Undefined            2       0      0
distribution_channel = Undefined      5       0      0
agent NULL (no agent)              6960    3716   5664
company NULL (no company)         46123   28093  38377
country missing                     284     121     83
country = CN (2-letter code)        279     362    638
country = CHN                       270     295    434
children missing                      4       0      0

Meal codes by hotel (all periods):
hotel      City Hotel  Resort Hotel
meal                               
BB              62305         30005
FB                 44           754
HB               6417          8046
SC              10564            86
Undefined           0          1169

Country code lengths: {2: 1279, 3: 117623}


**Observations**

- `meal`: Resort Hotel records "no meal" as `Undefined` (1,169 rows) and City Hotel as `SC` (10,564 rows; Resort uses `SC` only 86 times). The article defines both as "no meal package", so this is one category under two hotel-specific codes. → **C1: recode `Undefined` to `SC`.**
- `agent` / `company`: NULL means no agent or no company, not an unknown value. → **C2: store IDs as text and label NULL as `none`.**
- `country`: 488 missing in total. `CN` (1,279 rows) is the only 2-letter code and sits next to `CHN` (999 rows). `CN` is probably China's 2-letter ISO code, but nothing in the source confirms that. `country` is Bin 3 and never a predictor. → **C3: label missing as `unknown`; do not merge `CN` into `CHN`.**
- `children`: 4 train rows missing. → **C4: fill with 0 and keep a flag.**
- `market_segment = Undefined` (2 rows) and `distribution_channel = Undefined` (5 rows) are all in train and neither field is a candidate. → No action.

None of C1–C4 touches the four current candidate fields. They matter only if Bin 2 fields are used later, for example in the diagnostic leakage run listed in `docs/COLUMNS.md`.

## 4. Impossible or unusual bookings

Structural checks across all three periods (no outcome column read), then details and train-only outcomes for the flagged groups.

In [4]:
validity_checks = {
    "zero guests": lambda d: d["guests"].eq(0),
    "children or babies but 0 adults": lambda d: d["adults"].eq(0) & d["guests"].gt(0),
    "zero nights (same-day stay)": lambda d: d["nights"].eq(0),
    "zero guests and zero nights": lambda d: d["guests"].eq(0) & d["nights"].eq(0),
    "reserved room type P": lambda d: d["reserved_room_type"].eq("P"),
    "10 or more adults": lambda d: d["adults"].ge(10),
    "ADR below 0": lambda d: d["adr"].lt(0),
    "ADR = 0": lambda d: d["adr"].eq(0),
    "ADR = 0 in Complementary segment": lambda d: d["adr"].eq(0) & d["market_segment"].eq("Complementary"),
    "ADR above 1,000": lambda d: d["adr"].gt(1000),
    "3 or more parking spaces": lambda d: d["required_car_parking_spaces"].ge(3),
}
print(count_by_split(validity_checks).to_string())

                                  train  choose  proof
zero guests                          49      64     67
children or babies but 0 adults      72      88     63
zero nights (same-day stay)         371     200    144
zero guests and zero nights          11      25     34
reserved room type P                  1       5      6
10 or more adults                    13       0      0
ADR below 0                           0       0      1
ADR = 0                            1065     439    455
ADR = 0 in Complementary segment    309     170    201
ADR above 1,000                       1       0      0
3 or more parking spaces              0       1      4


In [5]:
detail_columns = ["hotel", "arrival_date", "adults", "children", "babies", "nights",
                  "adr", "market_segment", "deposit_type", "reserved_room_type"]

print("Three largest ADR values in train:")
print(train.nlargest(3, "adr")[detail_columns].to_string())

print("\nNegative ADR (proof period; outcome columns not shown):")
print(proof.loc[proof["adr"].lt(0), detail_columns].to_string())

print("\nReserved room type P (all periods; outcome columns not shown):")
print(bookings.loc[bookings["reserved_room_type"].eq("P"), detail_columns].to_string())

print("\nTrain bookings with 10 or more adults:")
print(train.loc[train["adults"].ge(10), detail_columns + ["is_canceled"]].to_string())

Three largest ADR values in train:
              hotel arrival_date  adults  children  babies  nights      adr market_segment deposit_type reserved_room_type
48515    City Hotel   2016-03-25       2       0.0       0       1  5400.00  Offline TA/TO   Non Refund                  A
15083  Resort Hotel   2015-07-15       2       0.0       0       1   508.00      Corporate   No Deposit                  A
90280    City Hotel   2016-06-02       2       2.0       0       3   332.57      Online TA   No Deposit                  F

Negative ADR (proof period; outcome columns not shown):
              hotel arrival_date  adults  children  babies  nights   adr market_segment deposit_type reserved_room_type
14969  Resort Hotel   2017-03-05       2       0.0       0      10 -6.38         Groups   No Deposit                  A

Reserved room type P (all periods; outcome columns not shown):


              hotel arrival_date  adults  children  babies  nights  adr market_segment deposit_type reserved_room_type
4127   Resort Hotel   2016-02-15       0       0.0       0       0  0.0  Offline TA/TO   No Deposit                  P
9376   Resort Hotel   2016-11-21       0       0.0       0       0  0.0  Offline TA/TO   No Deposit                  P
60651    City Hotel   2016-11-17       0       0.0       0      15  0.0      Corporate   No Deposit                  P
61244    City Hotel   2016-12-06       0       0.0       0       0  0.0  Complementary   No Deposit                  P
61245    City Hotel   2016-12-06       0       0.0       0       0  0.0  Complementary   No Deposit                  P
61249    City Hotel   2016-12-06       0       0.0       0       0  0.0  Complementary   No Deposit                  P
65907    City Hotel   2017-04-10       0       0.0       0       0  0.0  Complementary   No Deposit                  P
65908    City Hotel   2017-04-10       0       0

In [6]:
# Train only: how do the flagged groups differ in outcome?
train_groups = {}
train_groups["all train bookings"] = pd.Series(True, index=train.index)
train_groups["zero nights"] = train["nights"].eq(0)
train_groups["zero guests"] = train["guests"].eq(0)
train_groups["ADR = 0"] = train["adr"].eq(0)
train_groups["10 or more adults"] = train["adults"].ge(10)

group_rows = []
for group_name, group_mask in train_groups.items():
    group_rows.append({
        "group": group_name,
        "bookings": int(group_mask.sum()),
        "cancellation_rate": train.loc[group_mask, "is_canceled"].mean(),
    })

group_summary = pd.DataFrame(group_rows)
print(group_summary.to_string(
    index=False,
    formatters={"cancellation_rate": "{:.1%}".format},
))

print("\nZero-night train bookings by final status:")
print(train.loc[train["nights"].eq(0), "reservation_status"].value_counts().to_string())

             group  bookings cancellation_rate
all train bookings     49157             35.8%
       zero nights       371              4.6%
       zero guests        49              8.2%
           ADR = 0      1065             11.1%
 10 or more adults        13            100.0%

Zero-night train bookings by final status:
reservation_status
Check-Out    354
No-Show       10
Canceled       7


**Observations**

- **ADR 5,400** (one City Hotel train row, 2016-03-25, `Non Refund`, 1 night, 2 adults) is over 10 times the next highest train ADR (508). It looks like an entry error, but no source rule confirms that. It is in train, ADR is not a candidate input, and train nights are not used to choose settings, so it changes nothing the project scores (§8).
- **Negative ADR** (-6.38) is one proof row: a 10-night Resort Hotel `Groups` booking. Setting it to 0 would move that night's stay value by less than EUR 1 (§8).
- **Zero-night bookings** (715 in total) are mostly real same-day stays: in train, 354 of 371 checked out, and only 4.6% cancelled. A same-day stay never occupies a room overnight, yet the contract counts it in that night's book and its stay value of 0 pulls the night's average down.
- **Zero-guest bookings** (180 in total) include all 12 room-type `P` rows. Those 12 have no guests and ADR 0, and 11 have zero nights, so they look like placeholder records.
- **13 train bookings with 10-55 adults** are all Resort Hotel, September-October 2015, ADR 0 and cancelled. They look like group-block placeholders rather than single-room bookings.
- `ADR = 0` is common and partly explained by the `Complementary` segment (free stays), so 0 alone is not an error.

These rows can be explained or are too few to matter. The proposal is to keep every row in both tables and flag it (**C5**). Section 8 shows what removing them would change.

## 5. Consistency checks

The first cell reads outcome fields and therefore uses **train only**. The week-number check is structural and uses all periods.

In [7]:
print("Target vs reservation status (train):")
print(pd.crosstab(train["reservation_status"], train["is_canceled"]).to_string())

status_date = pd.to_datetime(train["reservation_status_date"])
checked_out = train["reservation_status"].eq("Check-Out")
expected_checkout = train["arrival_date"] + pd.to_timedelta(train["nights"], unit="D")
gap_days = (status_date - expected_checkout)[checked_out].dt.days
print(f"\nCheck-outs on the expected date (arrival + nights): {gap_days.eq(0).sum():,} of {checked_out.sum():,}")
print(f"Check-outs before the expected date (early departures): {gap_days.lt(0).sum():,}")
print(f"Check-outs after the expected date: {gap_days.gt(0).sum():,}")

booking_date = train["arrival_date"] - pd.to_timedelta(train["lead_time"], unit="D")
cancelled = train["reservation_status"].eq("Canceled")
print(f"Status dates before the booking date: {status_date.lt(booking_date).sum():,}")
print(f"Cancellations dated after arrival: {status_date[cancelled].gt(train.loc[cancelled, 'arrival_date']).sum():,}")

Target vs reservation status (train):


is_canceled             0      1
reservation_status              
Canceled                0  16959
Check-Out           31572      0
No-Show                 0    626

Check-outs on the expected date (arrival + nights): 31,559 of 31,572
Check-outs before the expected date (early departures): 13
Check-outs after the expected date: 0
Status dates before the booking date: 0
Cancellations dated after arrival: 0


In [8]:
# Structural check on all periods: which week definition does arrival_date_week_number follow?
jan1_weekday = pd.to_datetime(bookings["arrival_date_year"].astype(str) + "-01-01").dt.dayofweek
sunday_offset = (jan1_weekday + 1) % 7  # days between the Sunday that starts week 1 and 1 January
sunday_week = (bookings["arrival_date"].dt.dayofyear - 1 + sunday_offset) // 7 + 1
iso_week = bookings["arrival_date"].dt.isocalendar().week.astype(int)

week = bookings["arrival_date_week_number"]
print(f"Rows where the week number differs from the ISO week: {week.ne(iso_week).sum():,}")
print(f"Rows where it differs from Sunday-start weeks (week 1 contains 1 January): {week.ne(sunday_week).sum():,}")

Rows where the week number differs from the ISO week: 64,054


Rows where it differs from Sunday-start weeks (week 1 contains 1 January): 0


**Observations**

- `is_canceled` matches `reservation_status` exactly in train: `Canceled` and `No-Show` are 1 and `Check-Out` is 0, as the contract says.
- Status dates never precede the booking date, and no cancellation is dated after arrival. 13 check-outs happen before arrival + nights, which reads as early departure, not an error.
- `arrival_date_week_number` is not an ISO week, but it matches Sunday-start weeks exactly. The field is internally consistent. If a calendar feature is ever needed, derive it from `arrival_date`.

No cleaning is needed here.

## 6. Exact repeated rows

Train only, using all 32 columns as notebook 02 did. The last lines compare the non-outcome columns across all periods, so no choose or proof outcome is read.

In [9]:
train_source = train[source_columns]
group_size = train_source.groupby(
    source_columns,
    dropna=False,
)[source_columns[0]].transform("size")  # rows sharing exactly the same values
repeat_after_first = train_source.duplicated()

print(f"Exact repeated rows after the first occurrence: {repeat_after_first.sum():,}")
print(f"Rows belonging to an identical group: {group_size.gt(1).sum():,}")
print(f"Largest identical group: {group_size.max()} rows")
print(f"Rows in identical groups of 10 or more: {group_size.ge(10).sum():,}")

print("\nRepeated rows by market segment:")
print(train.loc[repeat_after_first, "market_segment"].value_counts().to_string())
print("\nRepeated rows by deposit type:")
print(train.loc[repeat_after_first, "deposit_type"].value_counts().to_string())

print(f"\nCancellation rate of repeated rows: {train.loc[repeat_after_first, 'is_canceled'].mean():.1%}")
print(f"Cancellation rate of rows with no identical copy: {train.loc[group_size.eq(1), 'is_canceled'].mean():.1%}")
print(f"Training cancellation rate now: {train['is_canceled'].mean():.1%}; "
      f"after dropping repeats: {train.loc[~repeat_after_first, 'is_canceled'].mean():.1%}")

print("\nShare of bookings repeating an earlier row in every non-outcome column:")
for name, period in splits.items():
    print(f"  {name}: {period[structural_columns].duplicated().mean():.1%}")

Exact repeated rows after the first occurrence: 16,291
Rows belonging to an identical group: 19,763
Largest identical group: 150 rows
Rows in identical groups of 10 or more: 11,339

Repeated rows by market segment:
market_segment
Groups           8047
Offline TA/TO    5964
Online TA        1273
Corporate         654
Direct            327
Complementary      23
Aviation            3

Repeated rows by deposit type:
deposit_type
No Deposit    9243
Non Refund    7005
Refundable      43

Cancellation rate of repeated rows: 63.2%
Cancellation rate of rows with no identical copy: 20.5%
Training cancellation rate now: 35.8%; after dropping repeats: 22.2%

Share of bookings repeating an earlier row in every non-outcome column:


  train: 33.6%
  choose: 23.1%
  proof: 22.5%


**Observations**

- The repeats cluster in `Groups` and `Offline TA/TO` bookings, some in identical groups of up to 150 rows. That pattern fits multi-room group bookings exported one row per room. The file has no booking ID, so this cannot be proved.
- They are strongly linked to the target: 63.2% of repeated rows cancelled, against 20.5% of rows with no copy. Dropping them would cut the training cancellation rate from 35.8% to 22.2%, so a model fitted on deduplicated data would under-predict cancellations on the full book it is scored on.
- Removing them would also remove about 23% of the scored book in choose and proof (see the shares above). That book is what the contract scores.
- Identical rows share their outcome by construction, so these blocks cancel together. `contract.nightly_table`'s `spread` assumes bookings cancel independently and will understate the night-to-night uncertainty. Note this for the uncertainty-buffer step; it is not a cleaning action.

→ Proposal **C5** keeps every row and flags identical groups. This adds evidence for notebook 02's gate D1.

## 7. Candidate feature consistency and drift

The four provisional inputs are `hotel`, `is_repeated_guest`, `previous_cancellations`, and `previous_bookings_not_canceled`. The first two cells read no outcome. The third uses train outcomes only.

In [10]:
def has_prior_history(period):
    """Return True where either prior-booking count is above zero."""
    had_cancellations = period["previous_cancellations"].gt(0)
    had_completed_bookings = period["previous_bookings_not_canceled"].gt(0)
    return had_cancellations | had_completed_bookings
candidate_checks = {
    "not a repeated guest, but has prior bookings": lambda d: d["is_repeated_guest"].eq(0) & has_prior_history(d),
    "repeated guest, but no prior bookings": lambda d: d["is_repeated_guest"].eq(1) & ~has_prior_history(d),
}
print(count_by_split(candidate_checks).to_string())

share_rows = {}
for name, period in splits.items():
    share_rows[name] = {
        "prior cancellations > 0": period["previous_cancellations"].gt(0).mean(),
        "prior non-cancelled bookings > 0": period["previous_bookings_not_canceled"].gt(0).mean(),
        "repeated guest": period["is_repeated_guest"].mean(),
    }

shares = pd.DataFrame(share_rows)
print("\nShare of bookings by period:")
print(shares.to_string(float_format="{:.1%}".format))

                                              train  choose  proof
not a repeated guest, but has prior bookings   5571     445    221
repeated guest, but no prior bookings           273     139    178

Share of bookings by period:
                                  train  choose  proof
prior cancellations > 0           11.8%    1.3%   0.7%
prior non-cancelled bookings > 0   2.7%    2.9%   3.5%
repeated guest                     3.3%    2.7%   3.4%


In [11]:
quarter = bookings["arrival_date"].dt.to_period("Q")
had_prior_cancellation = bookings["previous_cancellations"].gt(0)
quarterly_share = had_prior_cancellation.groupby(quarter).mean()
print("Share of bookings with prior cancellations, by arrival quarter:")
print(quarterly_share.to_string(float_format="{:.1%}".format))

Share of bookings with prior cancellations, by arrival quarter:
arrival_date
2015Q3   23.8%
2015Q4   16.0%
2016Q1    8.2%
2016Q2    2.9%
2016Q3    1.3%
2016Q4    1.4%
2017Q1    0.9%
2017Q2    0.6%
2017Q3    0.6%
Freq: Q-DEC


In [12]:
# Train only: is the prior-cancellation pattern stable inside the training period?
train_half = pd.Series("2016 H1", index=train.index, name="half")
early_train_rows = train["arrival_date"] < "2016-01-01"
train_half.loc[early_train_rows] = "2015 H2"
has_prior_cancel = train["previous_cancellations"].gt(0)
history = has_prior_cancel.map({True: "prior cancellations > 0", False: "none"}).rename("history")
prior_outcome = (
    train.groupby([train_half, history])["is_canceled"]
    .agg(["count", "mean"])
    .rename(columns={"count": "bookings", "mean": "cancellation_rate"})
)
print(prior_outcome.to_string(formatters={"cancellation_rate": "{:.1%}".format}))

in_group = group_size[has_prior_cancel].gt(1)
print(f"\nTrain rows with prior cancellations: {has_prior_cancel.sum():,}")
print(f"  in an identical-row group: {in_group.sum():,} ({in_group.mean():.0%})")
print(f"  with a Non Refund deposit: {train.loc[has_prior_cancel, 'deposit_type'].eq('Non Refund').sum():,}")
print(f"  marked as not a repeated guest: {train.loc[has_prior_cancel, 'is_repeated_guest'].eq(0).sum():,}")

                                 bookings cancellation_rate
half    history                                            
2015 H2 none                        17561             21.4%
        prior cancellations > 0      4435             99.0%
2016 H1 none                        25787             31.9%
        prior cancellations > 0      1374             89.2%

Train rows with prior cancellations: 5,809
  in an identical-row group: 4,983 (86%)
  with a Non Refund deposit: 3,444
  marked as not a repeated guest: 5,259


**Observations**

This is the most important finding for the current model plan.

- 5,571 train rows say "not a repeated guest" but carry prior bookings. Most of them are rows with prior cancellations, and 86% of train rows with prior cancellations belong to identical-row groups. One possible explanation is that `previous_cancellations` sometimes counts earlier cancelled entries from the same group or agency block, not the guest's own history. That is a hypothesis; the source does not confirm it.
- The field drifts sharply. 11.8% of train bookings have prior cancellations, compared with 1.3% in choose and 0.7% in proof. By quarter, the share falls from 23.8% (2015 Q3) to under 1% (2017).
- The pattern weakens inside train itself. Bookings with prior cancellations cancelled 99.0% of the time in 2015 H2 and 89.2% in 2016 H1.

The values may be accurate records, so changing them is not justified. The risk is that a model fitted on train relies on a pattern that has almost disappeared by the choose period. → Proposal **C6**: no value change. In Step 5, compare each candidate with and without `previous_cancellations` on choose. Choose outcomes are meant for this kind of method comparison. The contract also allows refitting on train + choose before the single proof run, provided every chosen setting is kept. C6 rests on the train and choose figures; the proof share is shown for context and is not used to decide. This adds evidence for notebook 02's gate D4.

## 8. What cleaning would do to the scored hotel-nights

The contract scores one hotel on one arrival night. The book is every booking due that night, and stay value V = mean(`adr` x `nights`). Removing a row changes the book, changing ADR changes V, and rule 5 requires every candidate to be scored on the same hotel-nights. Book sizes and V are computed directly here instead of with `contract.nightly_table`, which would also read `is_canceled`.

In [13]:
impact_rows = []
for name, period in splits.items():
    night_keys = [period["hotel"], period["arrival_date"]]
    nights_with = pd.DataFrame({
        "zero_night": period["nights"].eq(0),
        "zero_guest": period["guests"].eq(0),
    }).groupby(night_keys).any()
    impact_rows.append({
        "period": name,
        "hotel_nights": len(nights_with),
        "with_a_zero_night_booking": int(nights_with["zero_night"].sum()),
        "with_a_zero_guest_booking": int(nights_with["zero_guest"].sum()),
        "book_removed_if_repeats_dropped": period[structural_columns].duplicated().mean(),
    })
print(pd.DataFrame(impact_rows).to_string(index=False, formatters={"book_removed_if_repeats_dropped": "{:.1%}".format}))


def same_night(period, row_index):
    """Return every booking on the same hotel and arrival night as the given row."""
    row = period.loc[row_index]
    return period[(period["hotel"] == row["hotel"]) & (period["arrival_date"] == row["arrival_date"])]


top_adr = train["adr"].idxmax()
night = same_night(train, top_adr)
stay = night["adr"] * night["nights"]
print(f"\nADR 5,400 night (train): {len(night)} bookings; stay value {stay.mean():.2f} with the row, "
      f"{stay.drop(top_adr).mean():.2f} without it")

negative_adr = proof.index[proof["adr"].lt(0)][0]
night = same_night(proof, negative_adr)
stay = night["adr"] * night["nights"]
stay_if_zero = stay.where(night["adr"].ge(0), 0)
print(f"Negative-ADR night (proof): {len(night)} bookings; stay value {stay.mean():.2f} as recorded, "
      f"{stay_if_zero.mean():.2f} with ADR set to 0")

period  hotel_nights  with_a_zero_night_booking  with_a_zero_guest_booking book_removed_if_repeats_dropped
 train           732                        231                         39                           33.6%
choose           368                        140                         44                           23.1%
 proof           486                         93                         51                           22.5%

ADR 5,400 night (train): 182 bookings; stay value 377.73 with the row, 349.99 without it
Negative-ADR night (proof): 83 bookings; stay value 336.69 as recorded, 337.46 with ADR set to 0


**Observations**

- Zero-night bookings sit on many hotel-nights in every period. The contract defines the book as every booking due to arrive, so they stay in the scored book (rule 1). Whether a same-day stay should count toward a night's room oversell is a question for the client, not a cleaning step.
- Dropping repeated rows would remove about 23% of the book in choose and proof, which changes what the client's rule scores.
- The two ADR outliers barely move V: about EUR 28 on one train night that is never scored for decisions, and under EUR 1 on one proof night.

Conclusion: leave the **scored book** as the contract defines it, and do not remove rows from the **training table** either, because the model has to predict the same population it is scored on.

## 9. Cleaning plan: proposals and evidence

| # | Issue | Rows affected (train / choose / proof) | Proposed action | Why | Changes current candidate inputs? | Informs 02 gate |
|---|---|---|---|---|---|---|
| C1 | `meal = Undefined` | 548 / 208 / 413 | Recode to `SC` | Article defines Undefined/SC as one "no meal" category; only Resort uses `Undefined` | No | D2 |
| C2 | NULL `agent` / `company`; IDs stored as decimals | agent 6,960 / 3,716 / 5,664; company 46,123 / 28,093 / 38,377 | Store IDs as text; NULL becomes `none` | Article: NULL means not applicable | No | D2 |
| C3 | `country` missing | 284 / 121 / 83 | Label `unknown`; keep `CN` and `CHN` separate | Bin 3, never a predictor; `CN` meaning unconfirmed | No | D2 |
| C4 | `children` missing | 4 / 0 / 0 | Fill with 0; keep a flag | Most train rows have 0 children; tiny count | No | D2 |
| C5 | Repeated rows, zero nights or guests, room type `P`, 10+ adults, ADR 5,400 and -6.38 | see §4, §6 | Keep every row in the training table and the scored book; add `flag_` columns | Removal changes the book, the base rate (35.8% to 22.2%), and V; no source rule proves the rows are errors | No | D1, D3 |
| C6 | `previous_cancellations` drift and possible group counting | 11.8% / 1.3% / 0.7% of rows have prior cancellations | No value change; in Step 5 compare candidates with and without it on choose | Pattern weakens inside train and nearly disappears later | No (adds a test) | D4 |
| — | `Undefined` market segment / channel | 2 and 5 train rows | No action | Not candidates; tiny counts | — | — |
| — | Week number is not ISO | 0 mismatches with Sunday-start weeks | No action | Consistent with the date | — | — |

**Recommendation:** approve all six. C1–C4 are source-supported tidy-ups that only matter if Bin 2 or 3 fields are used later. C5 and C6 are the decisions that affect the MVP.

## 10. Your decisions

Edit each value to `"approve"` or `"disapprove"`, then run this cell and the two below it. `"pending"` makes no choice. A disapproval applies nothing and records that the branch is paused for discussion; no alternative action is guessed.

In [14]:
cleaning_decisions = {
    "C1_meal_undefined_to_sc": "pending",
    "C2_agent_company_text_with_none": "pending",
    "C3_country_unknown_label": "pending",
    "C4_children_fill_zero": "pending",
    "C5_keep_all_rows_with_flags": "pending",
    "C6_test_prior_cancellations_on_choose": "pending",
}

invalid = {key: value for key, value in cleaning_decisions.items() if value not in {"approve", "disapprove", "pending"}}
if invalid:
    raise ValueError(f"Use approve, disapprove, or pending: {invalid}")
print(pd.Series(cleaning_decisions).to_string())

C1_meal_undefined_to_sc                  pending
C2_agent_company_text_with_none          pending
C3_country_unknown_label                 pending
C4_children_fill_zero                    pending
C5_keep_all_rows_with_flags              pending
C6_test_prior_cancellations_on_choose    pending


## 11. Build the cleaned working table

`clean_bookings` always adds `flag_` columns, which record the observations above without changing a value. It applies the C1–C4 recodes only when approved. It never removes a row and never changes the contract's scoring inputs. Flags read no outcome column, so applying it to all periods reveals nothing about proof outcomes.

In [15]:
def clean_bookings(df, decisions):
    """Return a cleaned copy of df. Flags are always added; recodes apply only when approved."""
    out = df.copy()

    # Flags: observations only. No value is changed and no row is removed.
    out["flag_zero_guests"] = out["guests"].eq(0)
    out["flag_children_without_adults"] = out["adults"].eq(0) & out["guests"].gt(0)
    out["flag_zero_nights"] = out["nights"].eq(0)
    out["flag_room_type_p"] = out["reserved_room_type"].eq("P")
    out["flag_ten_plus_adults"] = out["adults"].ge(10)
    out["flag_adr_negative"] = out["adr"].lt(0)
    out["flag_adr_above_1000"] = out["adr"].gt(1000)  # only the 5,400 row; next highest is 508
    out["flag_children_missing"] = out["children"].isna()
    out["flag_repeat_status_conflict"] = out["is_repeated_guest"].eq(0) & (
        out["previous_cancellations"].gt(0) | out["previous_bookings_not_canceled"].gt(0)
    )
    # Identical in every non-outcome column, so no outcome is read.
    out["flag_identical_group"] = out[structural_columns].duplicated(keep=False)

    # Recodes: applied only when approved.
    if decisions["C1_meal_undefined_to_sc"] == "approve":
        out["meal"] = out["meal"].replace("Undefined", "SC")
    if decisions["C2_agent_company_text_with_none"] == "approve":
        for column in ["agent", "company"]:
            id_values = out[column]
            text_values = []

            for value in id_values:
                if pd.isna(value):
                    text_values.append("none")
                else:
                    text_values.append(str(int(value)))

            out[column] = text_values
    if decisions["C3_country_unknown_label"] == "approve":
        out["country"] = out["country"].fillna("unknown")
    if decisions["C4_children_fill_zero"] == "approve":
        out["children"] = out["children"].fillna(0).astype("int64")

    return out

In [16]:
cleaned = clean_bookings(bookings, cleaning_decisions)
clean_train, clean_choose, clean_proof = contract.split_periods(cleaned)

# Safety checks: every row kept, raw table untouched, contract scoring inputs unchanged.
scoring_inputs = ["hotel", "arrival_date", "adr", "nights"]
assert len(cleaned) == len(raw_bookings)
assert bookings[source_columns].equals(raw_bookings)
assert cleaned[scoring_inputs].equals(bookings[scoring_inputs])
print(f"Checks passed: {len(cleaned):,} rows kept; raw table and scoring inputs unchanged.")

status_text = {
    "approve": "APPLIED",
    "disapprove": "PAUSED - not applied; discuss an alternative",
    "pending": "PENDING - not applied",
}
recode_descriptions = {
    "C1_meal_undefined_to_sc": "meal 'Undefined' -> 'SC'",
    "C2_agent_company_text_with_none": "agent/company IDs as text, NULL -> 'none'",
    "C3_country_unknown_label": "missing country -> 'unknown'",
    "C4_children_fill_zero": "missing children -> 0",
}
print("\nRecodes:")
for key, description in recode_descriptions.items():
    print(f"  {key}: {status_text[cleaning_decisions[key]]} ({description})")

decision_descriptions = {
    "C5_keep_all_rows_with_flags": "keep every row in training and scoring; use flags, not removal",
    "C6_test_prior_cancellations_on_choose": "Step 5 compares candidates with and without previous_cancellations on choose",
}
print("\nDecisions with no data change:")
for key, description in decision_descriptions.items():
    status = "APPROVED" if cleaning_decisions[key] == "approve" else status_text[cleaning_decisions[key]]
    print(f"  {key}: {status} ({description})")

flag_columns = [column for column in cleaned.columns if column.startswith("flag_")]
flag_counts = pd.DataFrame({
    "train": clean_train[flag_columns].sum(),
    "choose": clean_choose[flag_columns].sum(),
    "proof": clean_proof[flag_columns].sum(),
}).astype(int)
print("\nFlagged rows by period:")
print(flag_counts.to_string())
print("\nMeal codes in the cleaned train table:", clean_train["meal"].value_counts().to_dict())

Checks passed: 119,390 rows kept; raw table and scoring inputs unchanged.

Recodes:
  C1_meal_undefined_to_sc: PENDING - not applied (meal 'Undefined' -> 'SC')
  C2_agent_company_text_with_none: PENDING - not applied (agent/company IDs as text, NULL -> 'none')
  C3_country_unknown_label: PENDING - not applied (missing country -> 'unknown')
  C4_children_fill_zero: PENDING - not applied (missing children -> 0)

Decisions with no data change:
  C5_keep_all_rows_with_flags: PENDING - not applied (keep every row in training and scoring; use flags, not removal)
  C6_test_prior_cancellations_on_choose: PENDING - not applied (Step 5 compares candidates with and without previous_cancellations on choose)

Flagged rows by period:
                              train  choose  proof
flag_zero_guests                 49      64     67
flag_children_without_adults     72      88     63
flag_zero_nights                371     200    144
flag_room_type_p                  1       5      6
flag_ten_plus_a

## Summary

- **Nothing was removed or overwritten.** The raw table is unchanged, every row is kept, and the contract's scoring inputs are identical. The cleaned table adds `nights`, `guests`, and ten `flag_` columns. Recodes C1–C4 take effect only after you approve them.
- **Most "dirty" values are explainable:** a hotel-specific meal code, NULL meaning "not applicable", same-day stays, complimentary rooms, and placeholder rows. Two ADR outliers exist but barely move any scored night.
- **Removing rows would do harm.** Repeated rows behave like real multi-room group blocks with a very different cancellation rate. Dropping them would shift the training rate from 35.8% to 22.2% and remove about 23% of the choose and proof book.
- **The biggest risk is a feature, not a dirty value.** `previous_cancellations` is concentrated in 2015 group blocks and nearly disappears by choose and proof. Step 5 should test it with and without the field on choose (C6).
- **Not done here:** no model trained, no proof outcome read, and no choice recorded as final.

Next, after your decisions: record each approved action with its row counts in `docs/DECISIONS.md`, as `src/data.py` asks. Then move `clean_bookings` into the planned `src/features.py` so `src/backtest.py` and `src/nightly.py` use the same cleaning.